In [34]:
import zipfile
import os
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split

Download e carregamento dos dados

In [35]:
files.upload() #kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d ashyou09/global-data-center-and-ai-waterelectricity-usage

zip_name = "global-data-center-and-ai-waterelectricity-usage.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_path)

print("Arquivos extraídos:")
for f in os.listdir(extract_path):
    print(f)

Saving kaggle.json to kaggle (2).json
Dataset URL: https://www.kaggle.com/datasets/ashyou09/global-data-center-and-ai-waterelectricity-usage
License(s): apache-2.0
global-data-center-and-ai-waterelectricity-usage.zip: Skipping, found more recently modified local copy (use --force to force download)
Arquivos extraídos:
data_center_hybrid.csv


Análise exploratória

In [36]:
df = pd.read_csv("/content/dataset/data_center_hybrid.csv")

print(df.shape)

(126770, 14)


In [37]:
df.head()

,Year,Facility_ID,Facility_Name,Owner_Company,City,Country,Facility_Type,Estimated_Capacity_MW,PUE,Cooling_System_Type,WUE_L_per_kWh,Daily_Electricity_Usage_MWh,Daily_Water_Usage_Gallons,Surrounding_Water_Stress_Tier
0,2019,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.24,1.975,Evaporative,1.481,183.62,36362.94,Low
1,2020,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.36,1.967,Evaporative,1.459,254.34,49833.60,Low
2,2021,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.47,1.928,Evaporative,1.450,266.85,53026.35,Low
3,2022,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.59,1.897,Evaporative,1.413,199.14,39198.30,Low
4,2023,DC-D2763E00,NAP de las Americas Madrid,Terremark,Madrid,Spain,Enterprise/Standard,6.70,1.869,Evaporative,1.389,225.94,44366.48,Low


In [38]:
df.columns

Index(['Year', 'Facility_ID', 'Facility_Name', 'Owner_Company', 'City',
       'Country', 'Facility_Type', 'Estimated_Capacity_MW', 'PUE',
       'Cooling_System_Type', 'WUE_L_per_kWh', 'Daily_Electricity_Usage_MWh',
       'Daily_Water_Usage_Gallons', 'Surrounding_Water_Stress_Tier'],
      dtype='object')

In [39]:
X = df.drop(['Daily_Water_Usage_Gallons', 'Facility_ID', 'Facility_Name', 'Owner_Company', 'City'], axis=1)

y = df['Daily_Water_Usage_Gallons']

In [40]:
X.columns

Index(['Year', 'Country', 'Facility_Type', 'Estimated_Capacity_MW', 'PUE',
       'Cooling_System_Type', 'WUE_L_per_kWh', 'Daily_Electricity_Usage_MWh',
       'Surrounding_Water_Stress_Tier'],
      dtype='object')

In [41]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126770 entries, 0 to 126769
Data columns (total 9 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Year                           126770 non-null  int64  
 1   Country                        126770 non-null  object 
 2   Facility_Type                  126770 non-null  object 
 3   Estimated_Capacity_MW          126770 non-null  float64
 4   PUE                            126770 non-null  float64
 5   Cooling_System_Type            126770 non-null  object 
 6   WUE_L_per_kWh                  126770 non-null  float64
 7   Daily_Electricity_Usage_MWh    126770 non-null  float64
 8   Surrounding_Water_Stress_Tier  126770 non-null  object 
dtypes: float64(4), int64(1), object(4)
memory usage: 8.7+ MB


In [42]:
X.describe()

,Year,Estimated_Capacity_MW,PUE,WUE_L_per_kWh,Daily_Electricity_Usage_MWh
count,126770.000000,126770.000000,126770.000000,126770.000000,126770.000000
mean,2022.000000,23.075117,1.637746,0.820596,605.222368
std,2.000008,45.847195,0.190833,0.925481,1040.061940
min,2019.000000,1.000000,1.057000,0.001000,22.190000
25%,2020.000000,5.450000,1.518000,0.139000,167.432500
50%,2022.000000,9.870000,1.643000,0.232000,302.750000
75%,2024.000000,14.180000,1.787000,1.654000,451.370000
max,2025.000000,562.890000,2.000000,3.000000,14812.660000


In [43]:
X.isnull().sum()

,0
Year,0
Country,0
Facility_Type,0
Estimated_Capacity_MW,0
PUE,0
Cooling_System_Type,0
WUE_L_per_kWh,0
Daily_Electricity_Usage_MWh,0
Surrounding_Water_Stress_Tier,0


Pré-processamento


In [44]:
#machine learning não aceita texto diretamente
X = pd.get_dummies(X, columns=['Facility_Type', 'Cooling_System_Type', 'Country', 'Surrounding_Water_Stress_Tier'])

Separa dados para treino e teste

In [45]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1765, random_state=42)

print(f"Treino: {len(X_train)} porcentagem ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validação: {len(X_val)} porcentagem ({len(X_val)/len(X)*100:.1f}%)")
print(f"Teste: {len(X_test)} porcentagem ({len(X_test)/len(X)*100:.1f}%)")

Treino: 88735 porcentagem (70.0%)
Validação: 19019 porcentagem (15.0%)
Teste: 19016 porcentagem (15.0%)
